In [ ]:
import pandas
train = pandas.read_csv('/kaggle/input/competitions/titanic/train.csv')
train.head(5)


In [ ]:
train.tail(5)


In [ ]:
train.iloc[100:200]

In [ ]:
train.isna().sum()

In [ ]:
test = pandas.read_csv('/kaggle/input/competitions/titanic/test.csv')
rozwiazanie1 = pandas.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': 1
})
rozwiazanie1.to_csv('/kaggle/working/rozwiazanie1.csv', index=False)
rozwiazanie1.head()


In [ ]:
test = pandas.read_csv('/kaggle/input/competitions/titanic/test.csv')

rozwiazanie3 = pandas.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': (test['Sex'] == 'female').astype(int)
})

rozwiazanie3.to_csv('/kaggle/working/rozwiazanie3.csv', index=False)

rozwiazanie3.head()

In [ ]:
test = pandas.read_csv('/kaggle/input/competitions/titanic/test.csv')

czy_przezyl = []
for wiek in test['Age']:
    if wiek < 11:
        czy_przezyl.append(1)
    else:
        czy_przezyl.append(0)

rozwiazanie4 = pandas.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': czy_przezyl
})

rozwiazanie4.to_csv('/kaggle/working/rozwiazanie4.csv', index=False)

rozwiazanie4.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, accuracy_score

# Dane
df = pd.DataFrame({
    "Label": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"],
    "Age": [35, 66, 42, 22, 38, 1, 38, 30, 21, 44],
    "Fare": [26, 10.5, 52, 9, 80, 39, 90, 86.5, 73.5, 90],
    "Survived": [0, 0, 0, 0, 1, 1, 1, 1, 0, 0]
})

# =====================================
# 1. TWOJA PROSTA
# =====================================

# Twoja prosta: Fare = a_moja * Age + b_moja
a_moja = 1.6
b_moja = 2

# Czy punkt jest nad Twoją prostą?
df["Moja_prosta_Fare"] = a_moja * df["Age"] + b_moja
df["Moja_predykcja"] = (df["Fare"] >= df["Moja_prosta_Fare"]).astype(int)

# Błąd Twojej prostej jako klasyfikatora
mse_moja = mean_squared_error(df["Survived"], df["Moja_predykcja"])
accuracy_moja = accuracy_score(df["Survived"], df["Moja_predykcja"])


# =====================================
# 2. MODEL REGRESJI LINIOWEJ
# =====================================

# Model uczy się: Age, Fare -> Survived
X = df[["Age", "Fare"]]
y = df["Survived"]

model = LinearRegression()
model.fit(X, y)

# Score, czyli wynik modelu, np. 0.72
df["Score_modelu"] = model.predict(X)

# Zamiana score na klasę 0/1
df["Predykcja_modelu"] = (df["Score_modelu"] >= 0.5).astype(int)

# Błędy modelu
mse_model = mean_squared_error(df["Survived"], df["Score_modelu"])
accuracy_model = accuracy_score(df["Survived"], df["Predykcja_modelu"])

# Parametry modelu:
# Score = a_model * Age + b_model * Fare + c_model
a_model = model.coef_[0]
b_model = model.coef_[1]
c_model = model.intercept_

# =====================================
# 2a. PARAMETRY PROSTEJ OBLICZONEJ PRZEZ MODEL
# =====================================

# Granica decyzyjna modelu:
# Score = 0.5
#
# a_model * Age + b_model * Fare + c_model = 0.5
#
# Po przekształceniu:
# Fare = a_prosta_modelu * Age + b_prosta_modelu

if abs(b_model) < 1e-12:
    raise ValueError("Nie można zapisać prostej modelu jako Fare = a * Age + b, bo b_model jest równe 0.")

a_prosta_modelu = -a_model / b_model
b_prosta_modelu = (0.5 - c_model) / b_model


# =====================================
# 3. WYPISYWANIE WYNIKÓW
# =====================================

print("TWOJA PROSTA:")
print(f"Fare = {a_moja:.4f} * Age + {b_moja:.4f}")
print("Reguła: jeśli punkt jest nad Twoją prostą, przewidujemy Survived = 1")
print(f"MSE klasyfikacji = {mse_moja:.4f}")
print(f"Accuracy = {accuracy_moja:.4f}")

print("\nMODEL REGRESJI LINIOWEJ:")
print(f"Score = {a_model:.4f} * Age + {b_model:.4f} * Fare + {c_model:.4f}")
print("Reguła: jeśli Score >= 0.5, przewidujemy Survived = 1")
print(f"MSE score = {mse_model:.4f}")
print(f"Accuracy = {accuracy_model:.4f}")

print("\nPROSTA OBLICZONA PRZEZ MODEL:")
print("Jest to granica decyzyjna dla Score = 0.5")
print(f"Fare = {a_prosta_modelu:.4f} * Age + {b_prosta_modelu:.4f}")
print(f"a_prosta_modelu = {a_prosta_modelu:.4f}")
print(f"b_prosta_modelu = {b_prosta_modelu:.4f}")

print("\nSPOSÓB LICZENIA BŁĘDU MODELU:")
print("MSE = średnia z (prawdziwe Survived - Score_modelu)^2")

print("\nTABELA:")
print(df[[
    "Label",
    "Age",
    "Fare",
    "Survived",
    "Moja_predykcja",
    "Score_modelu",
    "Predykcja_modelu"
]])


# =====================================
# 4. RYSOWANIE WYKRESU
# =====================================

age_line = np.linspace(0, 70, 200)

# Twoja prosta
fare_moja_line = a_moja * age_line + b_moja

# Prosta modelu:
# Fare = a_prosta_modelu * Age + b_prosta_modelu
fare_model_line = a_prosta_modelu * age_line + b_prosta_modelu

colors = df["Survived"].map({0: "red", 1: "green"})

plt.figure(figsize=(10, 6))

# Punkty
plt.scatter(df["Age"], df["Fare"], c=colors, s=100)

# Etykiety A-J
for i, row in df.iterrows():
    plt.text(row["Age"] + 0.7, row["Fare"] + 0.7, row["Label"], fontsize=12)

# Twoja prosta
plt.plot(
    age_line,
    fare_moja_line,
    color="orange",
    linewidth=2,
    label=f"Moja prosta: Fare = {a_moja:.2f} * Age + {b_moja:.2f}"
)

# Prosta z modelu
plt.plot(
    age_line,
    fare_model_line,
    color="blue",
    linewidth=2,
    label=f"Model: Fare = {a_prosta_modelu:.2f} * Age + {b_prosta_modelu:.2f}"
)

# Czarna linia "ad hoc" — ręcznie podane dwa punkty
# linia która wydaje się najlepsza ale model jej nie znajdzie
# Ma duze bledy sredniokwadratowe dla istniejacych punktow
czarna_age = [18, 46]
czarna_fare = [3, 103]
plt.plot(
    czarna_age,
    czarna_fare,
    color="black",
    linewidth=4,
    label='Czarna linia "ad hoc"'
)

# Legenda punktów
plt.scatter([], [], color="red", label="Survived = 0")
plt.scatter([], [], color="green", label="Survived = 1")

plt.xlabel("Age")
plt.ylabel("Fare")
plt.title("Twoja prosta i prosta z modelu regresji liniowej")
plt.legend()
plt.grid(True)
plt.xlim(0, 75)
plt.ylim(0, 110)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression


# =====================================
# USTAWIENIA
# =====================================
# Kolumny: Age, Fare, Pclass, SibSp, Parch, Sex

CECHY = ["Age", "Fare"]       # tutaj zmieniasz kolumny
PLIK_WYNIKOWY = "lekcja2_rozwiazanie_1.csv"

# =====================================
# 1. Wczytujemy dane
# =====================================

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')


# =====================================
# 2. Jeśli podano Sex - zmieniamy na liczby
# =====================================

if "Sex" in CECHY:
    train["Sex"] = train["Sex"].map({
        "male": 0,
        "female": 1
    })

    test["Sex"] = test["Sex"].map({
        "male": 0,
        "female": 1
    })


# =====================================
# 3. Wybieramy cechy
# =====================================

X_train = train[CECHY].copy()
y_train = train["Survived"]

X_test = test[CECHY].copy()


# =====================================
# 4. Uzupełniamy braki danych
# =====================================

for kolumna in CECHY:
    mediana = X_train[kolumna].median()
    
    X_train[kolumna] = X_train[kolumna].fillna(mediana)
    X_test[kolumna] = X_test[kolumna].fillna(mediana)


# =====================================
# 5. Uczymy regresję liniową
# =====================================

model = LinearRegression()    # pusty model regresji liniowej
model.fit(X_train, y_train)


# =====================================
# 6. Wypisujemy wzór modelu
# =====================================

a = model.coef_[0]
b = model.coef_[1]
c = model.intercept_

print("Model regresji liniowej:")
print(f"score = {a:.4f} * {CECHY[0]} + {b:.4f} * {CECHY[1]} + {c:.4f}")

print()
print("Linia oddzielająca jest tam, gdzie score = 0.5")

# Granica decyzji:
# score = 0.5
# a * cecha_1 + b * cecha_2 + c = 0.5
#
# Po przekształceniu:
# b * cecha_2 = 0.5 - c - a * cecha_1
# cecha_2 = (-a / b) * cecha_1 + (0.5 - c) / b

print()
print("Parametry prostej oddzielającej:")

if abs(b) > 1e-12:
    m = -a / b
    q = (0.5 - c) / b

    print(f"{CECHY[1]} = {m:.4f} * {CECHY[0]} + {q:.4f}")
    print(f"współczynnik kierunkowy m = {m:.4f}")
    print(f"wyraz wolny q = {q:.4f}")

else:
    # Gdy b jest równe 0, prosta jest pionowa.
    # Wtedy nie da się jej zapisać jako y = m*x + q.
    x_pionowa = (0.5 - c) / a

    print("Prosta oddzielająca jest pionowa.")
    print(f"{CECHY[0]} = {x_pionowa:.4f}")


# =====================================
# 7. Rysujemy punkty i linię oddzielającą
# =====================================

plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    X_train[CECHY[0]],
    X_train[CECHY[1]],
    c=y_train,
    alpha=0.7
)

if abs(b) > 1e-12:
    x_values = np.linspace(
        X_train[CECHY[0]].min(),
        X_train[CECHY[0]].max(),
        200
    )

    y_line = m * x_values + q

    plt.plot(
        x_values,
        y_line,
        linewidth=3,
        label="linia oddzielająca"
    )

else:
    plt.axvline(
        x=x_pionowa,
        linewidth=3,
        label="linia oddzielająca"
    )

plt.xlabel(CECHY[0])
plt.ylabel(CECHY[1])
plt.title("Titanic: regresja liniowa dla dwóch cech")
plt.legend()

plt.show()


# =====================================
# 8. Przewidujemy wyniki dla test.csv
# =====================================

test_scores = model.predict(X_test)

test_predictions = (test_scores >= 0.5).astype(int)


# =====================================
# 9. Tworzymy plik do Kaggle
# =====================================

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

submission.to_csv(PLIK_WYNIKOWY, index=False)

print("Zapisano plik:", PLIK_WYNIKOWY)
print(submission.head())